# Day 1: Architecture, raw data and Bronze

**Modern Data Engineering for AI Systems · SDA-DSC-214**

You will inspect the synthetic transport data, choose an architecture, preserve raw deliveries in Bronze, and compare compute cost assumptions. This is the first stage of your final project.

**Your lab:** 45 minutes to run and check, 5 minutes to answer, 5 minutes to discuss, and 5 minutes to save.

[Today's guide](https://github.com/almiyead-rgb/masar-ai-data-engineering-ksu/blob/main/day01/README.md) · [Expected results](https://github.com/almiyead-rgb/masar-ai-data-engineering-ksu/tree/main/expected) · [Submission guide](https://github.com/almiyead-rgb/masar-ai-data-engineering-ksu/blob/main/submission/README.md)

## Before you run

Use a CPU runtime. Select the documented Python 3.12 Colab runtime before running the first cell. The setup checks the versions and prepares Java 17. If Colab asks to restart after package installation, reconnect and run the setup again. Keep one GitHub repository for all five days.

Run cells in order. A dependency check is not a completed lab. Compare the outputs produced below with the reference results after the real engines run.

In [ ]:
from pathlib import Path
import importlib.util, json, os, runpy, subprocess, sys
DAY = 1
# Keep your own fork's owner/name here if you want to load committed edits.
PROJECT_REPOSITORY = "almiyead-rgb/masar-ai-data-engineering-ksu"
NONINTERACTIVE = os.environ.get("MASAR_NONINTERACTIVE") == "1"
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
if IS_COLAB and not NONINTERACTIVE:
    ROOT = Path("/content/masar-project")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/" + PROJECT_REPOSITORY + ".git", str(ROOT)], check=True)
    if not (ROOT / "course.json").is_file():
        raise RuntimeError("The project folder is incomplete. Preserve it and inspect the setup guide.")
    os.chdir(ROOT)
else:
    ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                 if (p / "course.json").is_file() and (p / "src/masar").is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError("Open this notebook from the complete course repository.")
sys.path.insert(0, str(ROOT / "src"))
bootstrap = runpy.run_path(str(ROOT / "scripts/bootstrap_colab.py"))
environment = bootstrap["prepare"](install=IS_COLAB)
os.environ["JAVA_HOME"] = environment["java_home"]
print(json.dumps(environment, indent=2))
from masar.runtime import start_spark
from masar.workspace import require_fixed_dataset, completed_bronze_workspace, new_workspace, write_json
from masar.native_contracts import validate_stage_result
from masar.colab import restore_handoff, record_reflection, record_metrics, save_day, initialize_student_notes
SOURCE = ROOT / "data/masar-small-v1"
require_fixed_dataset(SOURCE)
print("Course source:", ROOT)
print("Dependency checks are complete. The following cells execute the real engines.")
print("Interactive Drive/download UI enabled:", IS_COLAB and not NONINTERACTIVE)


## Your saved project

Allow Colab to mount your own Drive. Your daily continuation archives are saved in `MyDrive/Masar_Project`. A new session restores the preceding day only when the local project has no saved workspace. Existing work is not overwritten.

If you uploaded a handoff ZIP manually, call `restore_handoff` with its path and the preceding day number. Keep earlier archives while you learn.

After restoration, this cell creates your lab-note prompts under `student/dayNN/` and the shared project notes under `student/`. Open the printed paths in Colab’s Files panel and add your own evidence. Existing files are preserved. Your daily question answer belongs in `student/dayNN/REFLECTION.md`; the final save cell includes all these files in both packages.

In [ ]:
DRIVE_FOLDER = None
if IS_COLAB and not NONINTERACTIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_FOLDER = Path("/content/drive/MyDrive/Masar_Project")
    DRIVE_FOLDER.mkdir(parents=True, exist_ok=True)
if DAY > 1:
    pointer = ROOT / "outputs/day01_bronze_success.json"
    if not pointer.exists():
        backup = DRIVE_FOLDER / f"day{DAY-1:02d}_handoff.zip" if DRIVE_FOLDER else None
        if backup is None or not backup.is_file():
            raise FileNotFoundError(
                f"Restore your day{DAY-1:02d}_handoff.zip before continuing. "
                "Use restore_handoff(ROOT, Path('/path/to/your/archive.zip'), expected_day=DAY-1).")
        print(json.dumps(restore_handoff(ROOT, backup, expected_day=DAY-1), indent=2))
    WORK = completed_bronze_workspace(ROOT)
    print("Continue your project:", WORK.relative_to(ROOT))
else:
    pointer = ROOT / "outputs/day01_bronze_success.json"
    if pointer.exists():
        existing = completed_bronze_workspace(ROOT)
        if (existing / "reports/day02_silver.json").exists():
            raise RuntimeError("Your project has advanced beyond Day 1. Open the next day's notebook. Use a separate clean project for a complete replay.")
        WORK = existing
        REUSE_BRONZE = True
        print("Reusing your completed Bronze workspace without re-ingesting the same batches.")
    else:
        WORK = new_workspace(ROOT, "day01_bronze")
        REUSE_BRONZE = False
        print("New project:", WORK.relative_to(ROOT))

# Create prompts only after restoring your existing work. Never overwrite a note.
notes = initialize_student_notes(ROOT, DAY)
print(json.dumps(notes, indent=2))


## Five-minute activity

Place these examples in Bronze, Silver or Gold: an untouched trip delivery, one cleaned row per trip, and a city revenue summary.

<details><summary>Discuss the answer after trying</summary>

Bronze preserves the delivery. Silver provides the cleaned trip. Gold provides the city summary.

</details>

## Lab 01: inspect the source

Read the schema, counts and relationships. Keep the frozen data unchanged so your comparisons use the same inputs as the class.

In [ ]:
from masar.sources import load_sources, profile_sources
feeds = load_sources(SOURCE)
profile = profile_sources(SOURCE)
print(json.dumps({name: len(rows) for name, rows in feeds.items()}, indent=2))
print("Trip columns:", list(feeds["trips"][0]))
print(json.dumps(profile["profile"], indent=2))
print(json.dumps(profile["relations"], indent=2))
assert all(profile["checks"].values()), profile["checks"]
write_json(WORK / "reports/source_inspection.json", profile)

### Your turn: choose a business key

Try `driver_id` first, inspect the count, then use the correct trip key. Explain why the counts differ. The next cell includes a working starting choice you can inspect and change.

In [ ]:
business_key = "trip_id"
unique_business_keys = {row[business_key] for row in feeds["trips"]}
print("Chosen key:", business_key)
print("Distinct values:", len(unique_business_keys))
assert business_key == "trip_id", "Use the trip key before continuing to the next stage."

## Lab 01: preserve real Bronze deliveries

The same trip file is ingested with two distinct batch IDs. The raw history intentionally contains both deliveries. Spark writes and reads real Delta tables.

Expected after this exercise: **144 trip receipts, 72 unique trips, 6 drivers and 216 GPS events**.

In [ ]:
from masar.bronze import raw_frame, ingest_feed, verify_bronze
from masar.workspace import record_bronze_success
spark = start_spark(WORK)
try:
    raw_frame(spark, SOURCE, "trips").printSchema()
    if not REUSE_BRONZE:
        for feed in ("trips", "drivers", "gps_events"):
            print(ingest_feed(spark, SOURCE, WORK, feed, "base_001"))
        print(ingest_feed(spark, SOURCE, WORK, "trips", "replay_002"))
    bronze_report = verify_bronze(spark, SOURCE, WORK)
    record_bronze_success(ROOT, WORK)
    trips = spark.read.format("delta").load(str(WORK / "mini_lakehouse/bronze/trips"))
    metrics = {"bronze_trip_receipts": trips.count(), "unique_trips": trips.select("trip_id").distinct().count(),
               "drivers": bronze_report["counts"]["drivers"], "gps_events": bronze_report["counts"]["gps_events"]}
    print(json.dumps({"observed": metrics, "checks": bronze_report["checks"]}, indent=2))
    trips.select("trip_id", "_batch_id", "_source_file").orderBy("trip_id", "_batch_id").show(6, truncate=False)
finally:
    spark.stop()

## Lab 02: compare cost assumptions and storage reads

The cost model uses **hypothetical teaching units (TU)**, not cloud-provider prices. Compare an always-on scenario with a scheduled scenario. The native benchmark compares equivalent results from CSV and Delta; the small sample cannot establish a universal speed advantage.

In [ ]:
from masar.cost import cost_report
cost = cost_report()
print(json.dumps(cost["base"], indent=2))
write_json(WORK / "reports/cost_model_result.json", cost)
from masar.benchmark import benchmark
spark = start_spark(WORK)
try:
    benchmark_report = benchmark(spark, SOURCE, WORK, repetitions=4)
    validate_stage_result("lab02_scan", benchmark_report)
    print(json.dumps(benchmark_report["expected_and_observed_aggregate"], indent=2))
    print(json.dumps(benchmark_report["measurements"], indent=2))
finally:
    spark.stop()
record_metrics(ROOT, DAY, metrics)

## End-of-lab question

Bronze contains 144 trip receipt rows, but only 72 unique trip IDs. How many business trips do you have, and why should the raw receipt history be retained?

Write your answer and one observed value before opening the model answer. Your reflection is part of the project, worth one point when it explains your own run.

In [ ]:
MY_ANSWER = ""  # Write your own explanation before viewing the model answer.
MY_EVIDENCE = ""  # Include a value you observed and the report or cell that produced it.
reflection_path = record_reflection(ROOT, DAY, 'Bronze contains 144 trip receipt rows, but only 72 unique trip IDs. How many business trips do you have, and why should the raw receipt history be retained?', MY_ANSWER, MY_EVIDENCE)
print("Your reflection:", reflection_path.relative_to(ROOT))
if not MY_ANSWER.strip():
    print("Reflection pending: complete the answer in the file before submitting.")

<details><summary>Model answer: open after your attempt</summary>

There are 72 unique business trips and 144 delivery receipts. The same source was delivered twice. Bronze retains the source and arrival history so Silver can apply an explicit business-key rule without hiding how the data arrived.

Keep your first answer. Add a short correction in your reflection file if discussion changed your understanding.

</details>

## Save this day

1. Run the save cell below after all Spark sessions have stopped. It creates two packages.
2. In Colab use **File > Save a copy in GitHub**, select your repository and save this notebook as `day01/STUDENT.ipynb`. Keep your outputs.
3. Download the small `day01_github.zip`, unzip it on your computer, and upload its `student` and updated `day02/dbt` files to the same repository. Use a commit message such as `Complete Day 1`.
4. Keep the full handoff ZIP on Drive for your next session. It includes all cumulative Delta files and the separate Day 2 dbt workspace.
5. Open GitHub and confirm the notebook, reflection and results are visible.

Saving a notebook does not save Colab runtime files. The helper never publishes automatically and never asks you to put a token in a cell.

Before this cell, save your edits to the lab notes, shared project notes and reflection. Files marked **PENDING** still need your own explanation and observed evidence. Do not submit the unchanged prompts as completed work.

In [ ]:
saved = save_day(ROOT, DAY, drive_folder=DRIVE_FOLDER)
print(json.dumps(saved, indent=2))
print("Next: save this executed notebook to your GitHub repository using the File menu.")
if IS_COLAB and not NONINTERACTIVE:
    from google.colab import files
    files.download(saved["github_bundle"])